# Household models of WASH deprivation in Nigeria

Put this notebook beside **Nigeria_DHS_2013_2018_2024_Households_Climate.csv.gz** and select **Run → Run All Cells** in JupyterLab. Keep the household file compressed. If packages are missing, run **%pip install pandas numpy scipy scikit-learn xgboost matplotlib pillow python-docx** in a code cell and restart the kernel.

Results are saved in **wash_household_model_outputs**. For automatic Word 97–2003 export, install LibreOffice Writer. The notebook also saves a DOCX, which you can open in desktop Word and save as **Word 97–2003 Document (*.doc)** if the converter is unavailable.

This is the first household modelling comparison. It uses the main **2018 and 2024** outcome. The 2013 round stays in the master dataset and its eligibility is shown in the audit. The main model uses households with known water, sanitation and hygiene outcomes. The target is deprivation in at least two of those domains.


In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
from tempfile import TemporaryDirectory

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, brier_score_loss, log_loss, roc_auc_score
from sklearn.preprocessing import StandardScaler
from xgboost import DMatrix, XGBClassifier

ROOT = Path.cwd()
INPUT = ROOT / "Nigeria_DHS_2013_2018_2024_Households_Climate.csv.gz"
if not INPUT.is_file():
    INPUT = ROOT / "climate_stack_outputs" / INPUT.name
if not INPUT.is_file():
    raise FileNotFoundError("Put the compressed climate-linked household file beside the notebook.")
OUTPUT = ROOT / "wash_household_model_outputs"
OUTPUT.mkdir(exist_ok=True)
KEY = ["survey_round","hv001","hv002"]
TARGET = "wash_dep_2plus"
CLIMATE = ["rain_mm_3m","temp_mean_c_3m"]
DEMOGRAPHIC = ["rural","head_female","head_age_years_capped","usual_household_size","nonwash_asset_count"]
FEATURE_SETS = {"Household only": DEMOGRAPHIC, "Household and climate": DEMOGRAPHIC + CLIMATE}
NAMES = dict(zip(DEMOGRAPHIC + CLIMATE, ["Rural residence","Female household head","Age of household head",
             "Household size","Non-WASH asset count","Rainfall over 3 months","Mean temperature over 3 months"]))
RAW = ["head_age_years_capped","usual_household_size","nonwash_asset_count",*CLIMATE]
columns = [*KEY,"state","zone","residence","head_sex","hh_weight","wash_domains_known",TARGET,*RAW]
d = pd.read_csv(INPUT,usecols=columns,low_memory=False)
if d.groupby("survey_round").size().to_dict() != {2013:38522,2018:40427,2024:40047}:
    raise ValueError("Unexpected survey counts. Select the combined household output from the stack notebook.")
if d[KEY].isna().any().any() or d.duplicated(KEY).any():
    raise ValueError("Missing or duplicate household keys.")
if not np.isfinite(d.hh_weight).all() or not d.hh_weight.gt(0).all():
    raise ValueError("Household weights must be positive and finite.")
if not d[TARGET].dropna().isin([0,1]).all() or d.loc[d.survey_round.eq(2013),TARGET].notna().any():
    raise ValueError("The file does not use the verified main 2018–2024 target.")
main_rounds = d.survey_round.isin([2018,2024])
if not d.loc[main_rounds,TARGET].notna().eq(d.loc[main_rounds,"wash_domains_known"].eq(3)).all():
    raise ValueError("The target does not match the complete-case eligibility rule.")
if d[["state","zone"]].isna().any().any() or d.groupby("state").zone.nunique().ne(1).any():
    raise ValueError("Each state needs a consistent zone label.")
for column in RAW:
    d[column] = pd.to_numeric(d[column],errors="raise")
    if not np.isfinite(d[column].dropna()).all():
        raise ValueError(f"Non-finite predictor values in {column}.")
if not d.residence.dropna().isin(["Rural","Urban"]).all() or not d.head_sex.dropna().isin(["Female","Male"]).all():
    raise ValueError("Unexpected residence or head-sex labels.")
d["rural"] = d.residence.map({"Rural":1.,"Urban":0.})
d["head_female"] = d.head_sex.map({"Female":1.,"Male":0.})
analysis = d.loc[main_rounds & d[TARGET].notna()].copy()
analysis[TARGET] = analysis[TARGET].astype(int)
if analysis.groupby("survey_round").size().to_dict() != {2018:36821,2024:36412}:
    raise ValueError("Unexpected complete-case totals. Review the target and household input.")
audit = []
for year,g in d.groupby("survey_round"):
    known = g[TARGET].notna()
    audit.append({
        "survey_round":year, "households":len(g), "known_target":int(known.sum()),
        "missing_target":int((~known).sum()), "weighted_target_known_pct":100*g.loc[known,"hh_weight"].sum()/g.hh_weight.sum(),
        "rainfall_missing_among_known":int(g.loc[known,CLIMATE[0]].isna().sum()),
        "temperature_missing_among_known":int(g.loc[known,CLIMATE[1]].isna().sum()),
    })
audit = pd.DataFrame(audit)
audit.to_csv(OUTPUT/"Household_Model_Input_QC.csv",index=False)
display(audit)


## 1. Compare the models on held-out households

Four prespecified models combine two learners (logistic regression and shallow boosted trees) with two feature sets (household predictors alone and the same predictors plus rainfall and temperature). Features are restricted to the variables listed in the code. State, zone, survey round, cluster identifiers, WASH variables and the standard DHS wealth index are excluded from predictors. The non-WASH asset count is a study proxy for household resources. Environmental layers beyond rainfall and temperature can be added once they are linked.

Geographic evaluation holds out one of the six geopolitical zones at a time, across both rounds. Temporal evaluation fits on 2018 and tests on 2024 using the climate covariates observed for those households. The two evaluations answer separate questions. Training uses the household weights rescaled to average one within each training split. Test metrics use the original household weights. Missing predictors receive training medians within each split. The two feature sets use exactly the same households. Model settings are fixed before testing.

AUC and average precision measure ranking. Brier score and log loss evaluate predicted probabilities. Calibration compares predicted probability with observed deprivation. Scores are weighted descriptive metrics and the notebook does not compute survey-design confidence intervals.


In [ ]:
ZONES = ["North Central","North East","North West","South East","South South","South West"]
if set(analysis.zone) != set(ZONES):
    raise ValueError("The six geopolitical zones are required for the geographic evaluation.")

def fit_predict(train,test,features,learner):
    medians = train[features].median()
    if medians.isna().any():
        raise ValueError(f"A training feature is entirely missing: {medians[medians.isna()].index.tolist()}")
    xtrain = train[features].fillna(medians)
    xtest = test[features].fillna(medians)
    weight = train.hh_weight / train.hh_weight.mean()
    if learner == "Logistic":
        scaler = StandardScaler().fit(xtrain)
        model = LogisticRegression(C=1,max_iter=1000)
        model.fit(scaler.transform(xtrain),train[TARGET],sample_weight=weight)
        predicted = model.predict_proba(scaler.transform(xtest))[:,1]
    else:
        model = XGBClassifier(n_estimators=180,max_depth=3,learning_rate=.05,
                              min_child_weight=12,subsample=.9,colsample_bytree=.9,
                              tree_method="hist",random_state=20241111,n_jobs=2,
                              objective="binary:logistic",eval_metric="logloss")
        model.fit(xtrain,train[TARGET],sample_weight=weight)
        predicted = model.predict_proba(xtest)[:,1]
    return predicted,model,xtest

def summarize(test,predicted):
    y,w = test[TARGET],test.hh_weight
    if y.nunique() != 2:
        raise ValueError("The holdout must contain both outcome classes.")
    if not np.isfinite(predicted).all() or not ((predicted>=0)&(predicted<=1)).all():
        raise ValueError("Invalid predicted probabilities.")
    return {
        "n_test":len(test),"observed_prevalence":np.average(y,weights=w),
        "mean_prediction":np.average(predicted,weights=w),
        "auc":roc_auc_score(y,predicted,sample_weight=w),
        "average_precision":average_precision_score(y,predicted,sample_weight=w),
        "brier":brier_score_loss(y,predicted,sample_weight=w),
        "log_loss":log_loss(y,predicted,sample_weight=w,labels=[0,1]),
    }

scores, calibration_rows = [], []
temporal_train = analysis.loc[analysis.survey_round.eq(2018)]
temporal_test = analysis.loc[analysis.survey_round.eq(2024)]
splits = [(zone,analysis.loc[analysis.zone.ne(zone)],analysis.loc[analysis.zone.eq(zone)]) for zone in ZONES]
for zone,train,test in splits:
    if set(train.state) & set(test.state):
        raise ValueError(f"State leakage in the {zone} fold.")
assert not set(temporal_train.index) & set(temporal_test.index)
splits_qc = pd.DataFrame([
    {"holdout":zone,"n_train":len(train),"n_test":len(test),"held_out_states":test.state.nunique()}
    for zone,train,test in splits
] + [{"holdout":"2024 temporal","n_train":len(temporal_train),"n_test":len(temporal_test),
      "held_out_states":temporal_test.state.nunique()}])

for learner in ("Logistic","Boosted trees"):
    for feature_set,features in FEATURE_SETS.items():
        oof = pd.Series(np.nan,index=analysis.index)
        for zone,train,test in splits:
            predicted,_,_ = fit_predict(train,test,features,learner)
            if oof.loc[test.index].notna().any():
                raise ValueError("A household appears in more than one geographic holdout.")
            oof.loc[test.index] = predicted
            scores.append({"evaluation":"Geographic fold","holdout":zone,"learner":learner,
                           "features":feature_set,**summarize(test,predicted)})
        if oof.isna().any():
            raise ValueError("Missing geographic holdout predictions.")
        for label,group in [("All held-out zones",analysis), *[(str(y),g) for y,g in analysis.groupby("survey_round")]]:
            scores.append({"evaluation":"Geographic","holdout":label,"learner":learner,
                           "features":feature_set,**summarize(group,oof.loc[group.index].to_numpy())})

        predicted,model,xtest = fit_predict(temporal_train,temporal_test,features,learner)
        scores.append({"evaluation":"Temporal","holdout":"2024","learner":learner,
                       "features":feature_set,**summarize(temporal_test,predicted)})
        cal = temporal_test[[TARGET,"hh_weight"]].assign(predicted=predicted)
        cal["bin"] = pd.cut(cal.predicted,bins=np.linspace(0,1,11),include_lowest=True)
        for group,frame in cal.groupby("bin",observed=True):
            w=frame.hh_weight
            calibration_rows.append({
                "learner":learner,"features":feature_set,"bin":str(group),"n":len(frame),
                "mean_prediction":np.average(frame.predicted,weights=w),
                "observed":np.average(frame[TARGET],weights=w),
            })
        if learner == "Boosted trees" and feature_set == "Household and climate":
            explanation_model, explanation_input = model,xtest

scores = pd.DataFrame(scores)
calibration = pd.DataFrame(calibration_rows)
main_scores = scores.loc[scores.holdout.eq("All held-out zones") | scores.evaluation.eq("Temporal")].copy()
increment = main_scores.pivot(index=["evaluation","learner"],columns="features",values=["auc","brier","log_loss"])
increment = pd.DataFrame({
    "delta_auc":increment["auc"]["Household and climate"]-increment["auc"]["Household only"],
    "delta_brier":increment["brier"]["Household and climate"]-increment["brier"]["Household only"],
    "delta_log_loss":increment["log_loss"]["Household and climate"]-increment["log_loss"]["Household only"],
}).reset_index()
support = []
for feature in CLIMATE:
    train_values = temporal_train[feature].dropna()
    observed = temporal_test.loc[temporal_test[feature].notna()]
    below = observed[feature].lt(train_values.min())
    above = observed[feature].gt(train_values.max())
    support.append({"feature":feature,"train_min":train_values.min(),"train_max":train_values.max(),
                    "test_known":len(observed),"test_missing":len(temporal_test)-len(observed),
                    "test_weighted_outside_training_range_pct":
                        100*np.average(below|above,weights=observed.hh_weight) if len(observed) else np.nan})
support = pd.DataFrame(support)
display(main_scores.round(3))
display(increment.round(4))


## 2. Explain the boosted-tree predictions

TreeSHAP explains the prespecified household-and-climate boosted-tree model fitted on 2018, using the 2024 test households. It describes the fitted model even if another candidate predicts better. Contributions are measured on the log-odds scale. The feature ranking uses the weighted mean absolute contribution. The climate curves summarise mean SHAP contributions within bins of observed 2024 climate values, excluding households imputed for that feature.

These explanations show how predictors contributed to the model's predictions. Survey timing, correlated predictors and coverage differences can affect the patterns. The 2018 training-range check shows whether any observed 2024 climate values extend beyond that training range. The comparisons do not estimate causal effects or generate predictions for unsampled areas.

References include [structured cross-validation](https://doi.org/10.1111/ecog.02881) and the [XGBoost TreeSHAP API](https://xgboost.readthedocs.io/en/latest/python/python_api.html).


In [ ]:
matrix = DMatrix(explanation_input)
booster = explanation_model.get_booster()
contributions = booster.predict(matrix,pred_contribs=True)
margin = booster.predict(matrix,output_margin=True)
if not np.allclose(contributions.sum(axis=1),margin,atol=1e-5,rtol=1e-5):
    raise ValueError("TreeSHAP contributions do not reconstruct the prediction margin.")
weights = temporal_test.hh_weight.to_numpy()
shap_importance = pd.DataFrame({
    "feature":explanation_input.columns,
    "weighted_mean_abs_shap":np.average(np.abs(contributions[:,:-1]),axis=0,weights=weights),
    "weighted_mean_signed_shap":np.average(contributions[:,:-1],axis=0,weights=weights),
}).sort_values("weighted_mean_abs_shap",ascending=False)
dependence_rows = []
for feature in CLIMATE:
    column = explanation_input.columns.get_loc(feature)
    data = temporal_test[[feature,"hh_weight"]].assign(shap=contributions[:,column]).dropna(subset=[feature])
    if data[feature].nunique()<2:
        continue
    data["bin"] = pd.qcut(data[feature],q=10,duplicates="drop")
    for _,frame in data.groupby("bin",observed=True):
        w = frame.hh_weight
        dependence_rows.append({"feature":feature,"n":len(frame),"min":frame[feature].min(),
                                "max":frame[feature].max(),"weighted_feature":np.average(frame[feature],weights=w),
                                "weighted_mean_shap":np.average(frame.shap,weights=w)})
dependence = pd.DataFrame(dependence_rows,columns=["feature","n","min","max","weighted_feature","weighted_mean_shap"])
for filename,table in {
    "Household_Validation_Splits.csv":splits_qc,
    "Household_Model_Metrics.csv":scores,
    "Household_Climate_Increment.csv":increment,
    "Household_Temporal_Calibration.csv":calibration,
    "Household_Temporal_Climate_Support.csv":support,
    "Household_TreeSHAP_Importance.csv":shap_importance,
    "Household_TreeSHAP_Climate_Curves.csv":dependence,
}.items():
    table.to_csv(OUTPUT/filename,index=False)
display(shap_importance.round(3))


## 3. Save the Word report and 600 dpi figures

Open **wash_household_model_outputs**. Start with the Word report, then check the validation metrics and climate increment CSVs. A positive delta AUC or negative delta Brier means the climate feature set improved that metric in the specified holdout. Interpret each validation design separately. The report includes the 2024 calibration plot, TreeSHAP feature importance and the two climate attribution curves.


In [ ]:
dpi = 600

def save_figure(figure, filename):
    path = OUTPUT / filename
    figure.savefig(path, format="jpeg", dpi=dpi, facecolor="white",
                   pil_kwargs={"quality": 95, "subsampling": 0})
    plt.close(figure)
    with Image.open(path) as image:
        if image.format != "JPEG" or image.info.get("dpi") != (dpi, dpi):
            raise ValueError(f"Could not verify JPEG resolution for {path.name}.")
    return path




colors = ["#176C87","#9E4827","#514A87","#38754C"]
with plt.rc_context({"font.size":9,"axes.spines.top":False,"axes.spines.right":False}):
    fig,ax = plt.subplots(figsize=(7,3.55),layout="constrained")
    ax.plot([0,1],[0,1],linestyle="--",color="#818A91",linewidth=1,label="Perfect calibration")
    for ((learner,features),frame),color in zip(calibration.groupby(["learner","features"],sort=False),colors):
        frame=frame.sort_values("mean_prediction")
        ax.plot(frame.mean_prediction,frame.observed,"o-",markersize=3,linewidth=1.3,
                color=color,label=f"{learner}, {features.lower()}")
    ax.set(xlim=(0,1),ylim=(0,1),xlabel="Mean predicted probability",ylabel="Observed weighted proportion",
           title="Calibration in the 2024 temporal holdout")
    ax.legend(loc="upper left",frameon=False,fontsize=7)
    ax.grid(color="#E4E7EA")
    calibration_fig=save_figure(fig,"Figure_1_Temporal_Calibration_600dpi.jpg")

    fig,ax=plt.subplots(figsize=(7,3.1),layout="constrained")
    order=shap_importance.sort_values("weighted_mean_abs_shap")
    ax.barh(order.feature.map(NAMES),order.weighted_mean_abs_shap,color="#176C87")
    ax.set(xlabel="Weighted mean absolute SHAP value (log odds)",title="TreeSHAP feature importance in 2024")
    ax.grid(axis="x",color="#E4E7EA")
    ax.set_axisbelow(True)
    shap_fig=save_figure(fig,"Figure_2_TreeSHAP_Importance_600dpi.jpg")

    fig,axes=plt.subplots(1,2,figsize=(7,3.35),layout="constrained",sharey=True)
    for ax,feature,label,color in zip(axes,CLIMATE,["Three-month rainfall (mm)","Three-month mean temperature (°C)"],colors):
        frame=dependence.loc[dependence.feature.eq(feature)]
        ax.axhline(0,linestyle="--",linewidth=.9,color="#818A91")
        ax.plot(frame.weighted_feature,frame.weighted_mean_shap,"o-",markersize=3,color=color)
        if frame.empty:
            ax.text(.5,.5,"Insufficient variation",transform=ax.transAxes,ha="center")
        ax.set_xlabel(label)
        ax.grid(axis="y",color="#E4E7EA")
    axes[0].set_ylabel("Weighted mean SHAP value (log odds)")
    fig.suptitle("Climate contributions to 2024 predictions",fontsize=12)
    dependence_fig=save_figure(fig,"Figure_3_TreeSHAP_Climate_Curves_600dpi.jpg")
from docx import Document
from docx.enum.table import WD_TABLE_ALIGNMENT, WD_CELL_VERTICAL_ALIGNMENT
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.oxml import OxmlElement
from docx.oxml.ns import qn
from docx.shared import Inches, Pt, RGBColor

figure_width = 7
doc = Document()
doc.core_properties.title = "Household prediction of WASH deprivation in Nigeria"
doc.core_properties.author = ""
section = doc.sections[0]
section.page_width, section.page_height = Inches(8.5), Inches(11)
section.left_margin = section.right_margin = Inches(0.75)
section.top_margin, section.bottom_margin = Inches(0.7), Inches(0.7)

for style_name, size in [("Normal", 10.5), ("Title", 25), ("Heading 1", 17), ("Heading 2", 13), ("Caption", 9)]:
    style = doc.styles[style_name]
    style.font.name = "Calibri"
    style.font.size = Pt(size)
    style.font.color.rgb = RGBColor.from_string("000000")
    style.paragraph_format.space_before = Pt(5)
    style.paragraph_format.space_after = Pt(5)
    if style_name == "Title":
        border = style.element.get_or_add_pPr().find(qn("w:pBdr"))
        if border is not None:
            border.getparent().remove(border)

def xml(parent, tag, **attributes):
    element = OxmlElement(f"w:{tag}")
    for key, value in attributes.items():
        element.set(qn(f"w:{key}"), str(value))
    parent.append(element)
    return element

def add_table(caption, headers, rows, widths, text_columns, font_size=9.5):
    doc.add_paragraph(caption, style="Caption").paragraph_format.keep_with_next = True
    table = doc.add_table(rows=0, cols=len(headers))
    table.alignment = WD_TABLE_ALIGNMENT.CENTER
    table.autofit = False
    for column, width in zip(table.columns, widths):
        column.width = Inches(width)
    props = table._tbl.tblPr
    borders = xml(props, "tblBorders")
    for edge in ("top", "left", "bottom", "right", "insideH", "insideV"):
        xml(borders, edge, val="single", sz=4, color="D9D9D9")
    margins = xml(props, "tblCellMar")
    for edge, value in (("top", 16), ("bottom", 16), ("left", 85), ("right", 85)):
        xml(margins, edge, w=value, type="dxa")

    for i, values in enumerate([headers, *rows]):
        row = table.add_row()
        row_props = row._tr.get_or_add_trPr()
        xml(row_props, "cantSplit")
        if i == 0:
            xml(row_props, "tblHeader")
        for j, (cell, value, width) in enumerate(zip(row.cells, values, widths)):
            cell.width = Inches(width)
            cell.vertical_alignment = WD_CELL_VERTICAL_ALIGNMENT.CENTER
            if i == 0 or i % 2 == 0:
                xml(cell._tc.get_or_add_tcPr(), "shd", fill="203A50" if i == 0 else "F2F6F9")
            paragraph = cell.paragraphs[0]
            paragraph.alignment = WD_ALIGN_PARAGRAPH.LEFT if j in text_columns else WD_ALIGN_PARAGRAPH.CENTER
            paragraph.paragraph_format.space_after = Pt(0)
            paragraph.paragraph_format.space_before = Pt(0)
            paragraph.paragraph_format.line_spacing = 1.05
            run = paragraph.add_run(str(value))
            run.bold = i == 0
            run.font.size = Pt(font_size)
            if i == 0:
                run.font.color.rgb = RGBColor.from_string("FFFFFF")
    return table

def add_figure(path, caption, description):
    paragraph = doc.add_paragraph()
    paragraph.paragraph_format.space_before = Pt(7)
    paragraph.paragraph_format.keep_with_next = True
    paragraph.alignment = WD_ALIGN_PARAGRAPH.CENTER
    shape = paragraph.add_run().add_picture(str(path), width=Inches(figure_width))
    shape._inline.docPr.set("descr", description)
    doc.add_paragraph(caption, style="Caption")



doc.add_paragraph("Household prediction of WASH deprivation in Nigeria",style="Title")
doc.add_paragraph("Nigeria Demographic and Health Survey rounds 2018 and 2024")
doc.add_paragraph(
    f"The comparison includes {len(analysis):,} households with known WASH outcomes. It tests whether adding "
    "rainfall and temperature improves predictions from household characteristics. Geographic evaluation holds "
    "out one zone at a time. Temporal evaluation trains on 2018 and tests on 2024."
)
doc.add_heading("Validation performance",level=1)
rows=[[r.evaluation,r.learner,r.features,f"{r.n_test:,}",f"{r.auc:.3f}",
       f"{r.average_precision:.3f}",f"{r.brier:.3f}",f"{r.log_loss:.3f}"]
      for r in main_scores.itertuples(index=False)]
add_table("Table 1. Weighted held-out performance",
          ["Holdout","Learner","Predictors","Test n","AUC","AP","Brier","Log loss"],
          rows,[.9,1.,1.55,.7,.65,.65,.7,.85],{0,1,2},font_size=9)
rows=[[r.evaluation,r.learner,f"{r.delta_auc:+.4f}",f"{r.delta_brier:+.4f}",f"{r.delta_log_loss:+.4f}"]
      for r in increment.itertuples(index=False)]
add_table("Table 2. Change after adding the climate predictors",
          ["Holdout","Learner","AUC difference","Brier difference","Log loss difference"],
          rows,[1.25,1.3,1.4,1.4,1.65],{0,1})
doc.add_paragraph(
    "AP is average precision. Higher AUC and AP indicate better ranking. Lower Brier scores and log loss "
    "indicate better probability predictions. Geographic results pool the six held-out zones."
)
doc.add_heading("Calibration and feature contributions",level=1).paragraph_format.page_break_before=True
add_figure(calibration_fig,"Figure 1. Weighted calibration by prediction bin in the 2024 temporal holdout.",
           "Four model calibration curves against the diagonal reference line.")
add_figure(shap_fig,"Figure 2. Weighted absolute TreeSHAP contributions in the 2024 temporal holdout.",
           "Bar chart of mean absolute SHAP contributions by predictor.")
doc.add_heading("Climate contributions and coverage",level=1).paragraph_format.page_break_before=True
add_figure(dependence_fig,"Figure 3. Mean SHAP contributions by observed climate value in the 2024 holdout.",
           "Two panels showing binned climate contributions on the log-odds scale.")
rows=[[NAMES[r.feature],f"{r.train_min:.2f} to {r.train_max:.2f}",f"{r.test_known:,}",
       f"{r.test_missing:,}",f"{r.test_weighted_outside_training_range_pct:.2f}%"]
      for r in support.itertuples(index=False)]
add_table("Table 3. Temporal climate predictor support",
          ["Feature","2018 range","2024 known","2024 missing","Outside range"],
          rows,[2.,1.45,1.1,1.1,1.35],{0})
doc.add_paragraph(
    "TreeSHAP describes the 2018-trained boosted-tree model with household and climate predictors. "
    "It summarises the model's predictions among 2024 households with a known WASH outcome. "
    "The outcome is incomplete for some households in both rounds. The notebook reports eligibility and "
    "predictor missingness separately. Temporal evaluation uses observed 2024 climate data."
)
def find_soffice():
    custom = os.environ.get("LIBREOFFICE_PATH")
    if custom and Path(custom).is_file():
        return custom
    candidates = [shutil.which("soffice"), shutil.which("libreoffice"),
                  "/Applications/LibreOffice.app/Contents/MacOS/soffice"]
    for variable in ("ProgramFiles", "ProgramFiles(x86)"):
        if base := os.environ.get(variable):
            candidates.append(str(Path(base) / "LibreOffice" / "program" / "soffice.com"))
    return next((candidate for candidate in candidates if candidate and Path(candidate).is_file()), None)


docx_path = OUTPUT / "Nigeria_DHS_WASH_Household_Model_Report.docx"
doc.save(docx_path)

def export_legacy_word(source, soffice):
    with TemporaryDirectory() as scratch:
        temp = Path(scratch).resolve()
        result = subprocess.run(
            [soffice, f"-env:UserInstallation={(temp / 'profile').as_uri()}", "--headless",
             "--convert-to", "doc:MS Word 97", "--outdir", str(temp), str(source.resolve())],
            capture_output=True, text=True, timeout=180,
        )
        converted = temp / source.with_suffix(".doc").name
        if result.returncode or not converted.is_file():
            raise RuntimeError(result.stderr or result.stdout or "Converter returned no file.")
        with converted.open("rb") as stream:
            if stream.read(8) != bytes.fromhex("D0CF11E0A1B11AE1"):
                raise ValueError("The converter did not create a binary Word 97–2003 document.")
        shutil.copyfile(converted, source.with_suffix(".doc"))

soffice = find_soffice()
if soffice:
    try:
        export_legacy_word(docx_path, soffice)
        print(f"Saved {docx_path.with_suffix('.doc').resolve()}")
    except (RuntimeError, ValueError, OSError, subprocess.TimeoutExpired) as error:
        print(f"Word .doc conversion failed: {error}")
        print("Open the saved .docx in desktop Word and save it as Word 97–2003 Document (*.doc).")
else:
    print("LibreOffice was unavailable. Open the saved .docx in desktop Word and save it as Word 97–2003 Document (*.doc).")


print(f"Saved the report, three verified 600 dpi JPEGs and CSV results in {OUTPUT.resolve()}.")
